# Protocolo de treino e seleção
Este notebook inspeciona a configuração e os artefatos. O treinamento fica em
`src/fraud_detection/train.py`; não dispara novamente ao abrir o notebook.

Execute na raiz:
```powershell
python -m fraud_detection.eda
python -m fraud_detection.train
```
São duas famílias: Random Forest e XGBoost, com/sem under+SMOTE; Dummy é só referência.
Imputação e amostragem ocorrem dentro de cada treino de fold. TEST não entra na seleção.
Threshold fixo em 0,5; não houve tuning de threshold/hiperparâmetros.

## Como interpretar as métricas

As tabelas abaixo têm rótulos descritivos. Percentuais são apenas uma forma de
exibir os valores: **0,000190 = 0,0190%**. Os CSVs conservam valores numéricos.

| Métrica | Pergunta que responde / denominador |
|---|---|
| Taxa de fraude na faixa | Entre as transações dessa faixa, quantas são fraude? = 100 × fraudes / transações da faixa. Não é erro do modelo. |
| Parcela das fraudes do TRAIN | Entre todas as fraudes do TRAIN, quantas estão nessa faixa? |
| Recall | Entre as fraudes reais, quantas o modelo detectou? = TP/(TP+FN). |
| Precision | Entre os alertas, quantos eram fraude real? = TP/(TP+FP). |
| F1 | Média harmônica de Precision e Recall, com peso simétrico. |
| F2 | Média harmônica que favorece Recall; objetivo das rodadas posteriores. |
| FP / FN | FP é legítima alertada; FN é fraude não detectada. TP/TN são acertos de fraude/legítima. |
| Taxa de falsos positivos | FP/(FP+TN): fração das legítimas que gerou alerta. |
| Parcela de alertas falsos | FP/(FP+TP) = 1−Precision. Tem denominador diferente da taxa de falsos positivos. |
| AP / curva PR | AP resume a curva Precision–Recall usando probabilidades; maior é melhor. Não equivale à área trapezoidal PR. |
| ROC-AUC | Resumo de ordenação fraude versus legítima; complementar à AP para classe rara. |
| Log Loss / Brier | Erros das probabilidades: menor é melhor. Não comprovam calibração. |
| Média e desvio de CV | Resumo entre folds temporais; desvio não é intervalo de confiança. |
| Correlação / importância | Correlação é associação com Class; importância de permutação é queda de AP na validação interna. Escalas distintas e sem causalidade. |

Se um segmento não tiver fraudes ou alertas, certas taxas ficam **Não definido**:
isso não significa taxa zero. Não some categorias de features distintas:
as mesmas transações reaparecem em cada feature.

Veja [o guia completo das métricas](../docs/metrics_guide.md), incluindo fórmulas,
unidades, outliers/IQR, assimetria, cardinalidade, diversidade OOF, threshold,
StandardScaler, SMOTE e limitações da validação.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Abra o notebook pela raiz do projeto ou pela pasta notebooks.")
METRICS = ROOT / "reports/metrics"
FIGURES = ROOT / "reports/figures"
EXPERIMENTS = ROOT / "reports/experiments"

def table(name, **kwargs):
    path = METRICS / name
    if not path.exists():
        raise FileNotFoundError(f"Artefato ausente: {path.name}. Execute os scripts conforme README.")
    return pd.read_csv(path, **kwargs)

def figure(name, width=1000):
    path = FIGURES / name
    if not path.exists():
        raise FileNotFoundError(f"Figura ausente: {path.name}. Execute os scripts conforme README.")
    display(Image(filename=str(path), width=width))

# Mantém os cálculos numéricos; traduz unidades e rótulos apenas ao exibir a tabela.
from fraud_detection.reporting import readable_table

def display_readable(frame):
    display(readable_table(frame))


In [2]:
selection = json.loads((EXPERIMENTS / "selection.json").read_text(encoding="utf-8"))
display(selection)
display_readable(pd.read_csv(EXPERIMENTS / "fold_manifest.csv"))
display_readable(table("model_comparison.csv").drop(columns="configuration"))

{'model': 'xgboost__none',
 'selection': 'maior AP média em CV temporal TRAIN',
 'threshold': 0.5,
 'config': {'data_path': 'data/raw/creditcard.csv',
  'seed': 42,
  'test_time_fraction': 0.2,
  'cv_splits': 3,
  'threshold': 0.5,
  'primary_metric': 'average_precision',
  'n_jobs': 2,
  'sampling_strategies': ['none', 'under_smote'],
  'under_ratio': 0.1,
  'smote_ratio': 0.6,
  'smote_neighbors': 3,
  'models': {'random_forest': {'n_estimators': 160,
    'max_depth': 14,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'criterion': 'gini',
    'class_weight': 'balanced_subsample'},
   'xgboost': {'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.0,
    'reg_lambda': 1.0,
    'min_child_weight': 3}}},
 'features': ['Time',
  'V1',
  'V2',
  'V3',
  'V4',
  'V5',
  'V6',
  'V7',
  'V8',
  'V9',
  'V10',
  'V11',
  'V12',
  'V13',
  'V14',
  'V15',
  'V16',
  'V17',
  'V18',
  'V19',
  'V2

,Fold temporal,fit_rows,validation_rows,fit_positive,validation_positive,fit_time_max,validation_time_min,validation_time_max
0,1,45.840,58.589,143,84,"42.612,0000","42.613,0000","69.254,0000"
1,2,104.429,51.704,227,105,"69.254,0000","69.255,0000","108.793,0000"
2,3,156.133,54.159,332,44,"108.793,0000","108.799,0000","138.230,0000"


,Receita,Origem,Limiar de alerta,AP (score),average_precision_std,F1 (%),F1 desvio (p.p.),Recall (%),Recall desvio (p.p.),Precision (%),Precision desvio (p.p.),ROC-AUC (score),roc_auc_std,Balanced Accuracy (%),Balanced Accuracy desvio (p.p.)
0,xgboost__none,CV temporal TRAIN,"0,5000","0,7689","0,0208",77.19%,2.34 p.p.,69.96%,7.09 p.p.,87.67%,11.23 p.p.,"0,9700","0,0193",84.97%,3.54 p.p.
1,random_forest__none,CV temporal TRAIN,"0,5000","0,7580","0,0325",77.78%,1.47 p.p.,67.93%,5.24 p.p.,91.80%,7.28 p.p.,"0,9390","0,0434",83.96%,2.62 p.p.
2,random_forest__under_smote,CV temporal TRAIN,"0,5000","0,7513","0,0354",58.22%,21.90 p.p.,80.40%,5.88 p.p.,50.87%,30.88 p.p.,"0,9661","0,0264",90.13%,2.90 p.p.
3,xgboost__under_smote,CV temporal TRAIN,"0,5000","0,7387","0,0084",35.40%,26.94 p.p.,83.19%,7.78 p.p.,26.76%,25.46 p.p.,"0,9661","0,0165",91.25%,3.71 p.p.
4,dummy,CV temporal TRAIN,"0,5000","0,0014","0,0006",0.00%,0.00 p.p.,0.00%,0.00 p.p.,0.00%,0.00 p.p.,"0,5000","0,0000",50.00%,0.00 p.p.


## Reprodutibilidade e decisões
O manifesto registra hash da base, versões, folds, seed, contagens e cobertura OOF.
Uma segunda execução na mesma saída é bloqueada para evitar sobrescrever TEST
silenciosamente. Use `--output reports/reproductions/run_02` para reproduzir o protocolo;
isso não cria um TEST novo. Depois de mudar decisões com base no TEST, obtenha outro
período externo para avaliação independente. Os diagnósticos completos ficam no notebook 03.

## Rodada posterior: ajuste de threshold
Execute `python -m fraud_detection.threshold` para GridSearchCV de limiar da
receita XGBoost com ênfase em Recall usando F2. A grade e a janela interna ficam
em `configs/threshold.yaml`; folds externos avaliam a política.
O novo artefato `models/recall_pipeline.joblib` é separado da primeira rodada.
Não há nova avaliação no TEST já visto. Resultados e limites estão no notebook 03
e em `reports/threshold_results.md`.